<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка-данных" data-toc-modified-id="Подготовка-данных-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка данных</a></span></li><li><span><a href="#Исследование-задачи" data-toc-modified-id="Исследование-задачи-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Исследование задачи</a></span></li><li><span><a href="#Борьба-с-дисбалансом" data-toc-modified-id="Борьба-с-дисбалансом-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Борьба с дисбалансом</a></span></li><li><span><a href="#Тестирование-модели" data-toc-modified-id="Тестирование-модели-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Тестирование модели</a></span></li><li><span><a 

# Отток клиентов

Из «Бета-Банка» стали уходить клиенты. Каждый месяц. Немного, но заметно. Банковские маркетологи посчитали: сохранять текущих клиентов дешевле, чем привлекать новых.

Нужно спрогнозировать, уйдёт клиент из банка в ближайшее время или нет. Вам предоставлены исторические данные о поведении клиентов и расторжении договоров с банком. 

Постройте модель с предельно большим значением *F1*-меры. Чтобы сдать проект успешно, нужно довести метрику до 0.59. Проверьте *F1*-меру на тестовой выборке самостоятельно.

Дополнительно измеряйте *AUC-ROC*, сравнивайте её значение с *F1*-мерой.

Источник данных: [https://www.kaggle.com/barelydedicated/bank-customer-churn-modeling](https://www.kaggle.com/barelydedicated/bank-customer-churn-modeling)

## Подготовка данных

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn import svm
from sklearn.utils import shuffle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score 


In [2]:
df = pd.read_csv('/datasets/Churn.csv')
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


In [3]:
df = df.dropna()
df = df.drop(['Surname'], axis=1)
df = df.drop(['CustomerId'], axis=1)
df = df.drop(['RowNumber'], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 9091 entries, 0 to 9998
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      9091 non-null   int64  
 1   Geography        9091 non-null   object 
 2   Gender           9091 non-null   object 
 3   Age              9091 non-null   int64  
 4   Tenure           9091 non-null   float64
 5   Balance          9091 non-null   float64
 6   NumOfProducts    9091 non-null   int64  
 7   HasCrCard        9091 non-null   int64  
 8   IsActiveMember   9091 non-null   int64  
 9   EstimatedSalary  9091 non-null   float64
 10  Exited           9091 non-null   int64  
dtypes: float64(3), int64(6), object(2)
memory usage: 852.3+ KB


После исследования данных мы удаляем пропуски в столбце Tenure (их нечем заменить), а так же избавляемся от фамилий, номеров столбцов и идентификаторов клиентов (они не влияют на вопрос задачи)

## Исследование задачи

In [4]:
df_ohe = pd.get_dummies(df, drop_first=True)
target = df_ohe['Exited']
features = df_ohe.drop(['Exited'], axis=1)
# features_train, features_valid, target_train, target_valid = train_test_split(
#     features, target, test_size=0.25, random_state=12345)

x, features_valid, y, target_valid = train_test_split(
    features, target, test_size=0.2, random_state=12345)
features_train, features_test, target_train, target_test = train_test_split(
    x, y, test_size=0.25, random_state=12345)

In [5]:
%%time
# Дерево
for depth in range(1, 8):
    model_d = DecisionTreeClassifier(max_depth=depth, random_state=12345)  
    model_d.fit(features_train, target_train)
    predictions_valid = model_d.predict(features_valid)
    print("max_depth =", depth, ": ", end='')
#     print(accuracy_score(target_valid, predictions_valid))
    print("F1:", f1_score(target_valid, predictions_valid))
    probabilities_valid = model_d.predict_proba(features_valid)
    probabilities_one_valid = probabilities_valid[:, 1]
    print("auc_roc:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))

max_depth = 1 : F1: 0.0
auc_roc: 0.6899243061396131
max_depth = 2 : F1: 0.5234248788368336
auc_roc: 0.7416409681338192
max_depth = 3 : F1: 0.3907563025210084
auc_roc: 0.7968526305952716
max_depth = 4 : F1: 0.41493775933609955
auc_roc: 0.8067974955611624
max_depth = 5 : F1: 0.5831960461285008
auc_roc: 0.8147892720306514
max_depth = 6 : F1: 0.5437956204379562
auc_roc: 0.8431782076441453
max_depth = 7 : F1: 0.585209003215434
auc_roc: 0.8365255583590319
CPU times: user 139 ms, sys: 0 ns, total: 139 ms
Wall time: 150 ms


In [6]:
%%time
# Лес с заданной глубиной

best_model = None
best_result = 0
for est in range(1, 27):
    model_f = RandomForestClassifier(random_state=12345, n_estimators=est)
    model_f.fit(features_train, target_train)
    predictions_valid = model_f.predict(features_valid)
    result = f1_score(target_valid, predictions_valid) 
    if result > best_result:
        best_model = model_f
        best_result = result

print("F1 наилучшей модели на валидационной выборке:", best_result)
probabilities_valid = best_model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc наилучшей модели:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))
print(best_model)

F1 наилучшей модели на валидационной выборке: 0.5512605042016807
auc_roc наилучшей модели: 0.8444593963181011
RandomForestClassifier(n_estimators=23, random_state=12345)
CPU times: user 2.64 s, sys: 15.7 ms, total: 2.65 s
Wall time: 2.67 s


In [7]:
%%time
# model_s = svm.SVC() 
# model_s.fit(features_train, target_train)
# predictions_valid = model_s.predict(features_valid)
# print("F1:", f1_score(target_valid, predictions_valid))

model = LogisticRegression(solver='liblinear', random_state=12345)
model.fit(features_train, target_train)
predicted_valid = model.predict(features_valid)
print("F1:", f1_score(target_valid, predicted_valid))
probabilities_valid = model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))

F1: 0.02624671916010499
auc_roc: 0.6620801794224839
CPU times: user 52.2 ms, sys: 16.6 ms, total: 68.8 ms
Wall time: 49.8 ms


После разбивки выборки на тренировачную, тестировачную и валидационную мы протестировали 3 модели - лес с заданной шлубиной, Дерево и Логистическую. Модель логистической регресии бесролезна без учета дисбаланса (f1 - 0.03, auc-roc - 0.66). Лес с заданной глубиной дал лучший результат при максимальной глубине в 23 - (f1 - 0.55, auc-roc - 0.84). Лучшие результаты Дерева: при глубине 5 и 7 (f1 в обоих случаях равно 0,58), auc-roc немного больше при глубине 7 (0,84 и 0,81).

## Борьба с дисбалансом

In [8]:
print(df['Exited'].value_counts())
print (1854/7237)

0    7237
1    1854
Name: Exited, dtype: int64
0.2561835014508774


Примерный дисбаланс нашей выборки 4 к 1.

In [9]:
# upsample
def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_upsampled = pd.concat([features_zeros] + [features_ones] * repeat)
    target_upsampled = pd.concat([target_zeros] + [target_ones] * repeat)
    
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345)
    
    return features_upsampled, target_upsampled

features_upsampled, target_upsampled = upsample(features_train, target_train, 4)



In [10]:
%%time
model = LogisticRegression(solver='liblinear', random_state=12345)
model.fit(features_upsampled, target_upsampled)
predicted_valid = model.predict(features_valid)
print("F1:", f1_score(target_valid, predicted_valid))
probabilities_valid = model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))


F1: 0.45688888888888896
auc_roc: 0.7155798523502478
CPU times: user 78.2 ms, sys: 20.7 ms, total: 98.9 ms
Wall time: 48.2 ms


In [11]:
%%time
# Дерево

for depth in range(1, 8):
    model_d = DecisionTreeClassifier(max_depth=depth, random_state=12345)  
    model_d.fit(features_upsampled, target_upsampled)
    predictions_valid = model_d.predict(features_valid)
    print("max_depth =", depth, ": ", end='')
#     print(accuracy_score(target_valid, predictions_valid))
    print("F1:", f1_score(target_valid, predictions_valid))
    probabilities_valid = model_d.predict_proba(features_valid)
    probabilities_one_valid = probabilities_valid[:, 1]
    print("auc_roc:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))

max_depth = 1 : F1: 0.4892412231030578
auc_roc: 0.6899243061396131
max_depth = 2 : F1: 0.5187637969094923
auc_roc: 0.7416409681338192
max_depth = 3 : F1: 0.5353418308227114
auc_roc: 0.7971161573684703
max_depth = 4 : F1: 0.4873524451939291
auc_roc: 0.8069516867582468
max_depth = 5 : F1: 0.5430463576158941
auc_roc: 0.8155555555555556
max_depth = 6 : F1: 0.5741728922091781
auc_roc: 0.8272096065788244
max_depth = 7 : F1: 0.5528134254689042
auc_roc: 0.8192000747593683
CPU times: user 190 ms, sys: 27.2 ms, total: 217 ms
Wall time: 270 ms


In [12]:
%%time
# Лес с заданной глубиной

best_model = None
best_result = 0
for est in range(1, 27):
    model_f = RandomForestClassifier(random_state=12345, n_estimators=est)
    model_f.fit(features_upsampled, target_upsampled) 
    predictions_valid = model_f.predict(features_valid)
    result = f1_score(target_valid, predictions_valid) 
    if result > best_result:
        best_model = model_f
        best_result = result

print("F1 наилучшей модели на валидационной выборке:", best_result)
probabilities_valid = best_model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc наилучшей модели:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))
print(best_model)

F1 наилучшей модели на валидационной выборке: 0.5772005772005772
auc_roc наилучшей модели: 0.8234940659751426
RandomForestClassifier(n_estimators=13, random_state=12345)
CPU times: user 3.58 s, sys: 8.59 ms, total: 3.58 s
Wall time: 3.6 s


Результаты работы логистической модели резко повысились после учета дисбаланса. (f1 - 0.46, auc-roc - 0.72). Однако другие модели дали практически идентичные результаты - лес: f1 - 0.55 vs 0.58, auc-roc - 0.84 vs 0.82 (оптимальная глубина изменилась с 23 до 13); дерево: f1 - 0.58 vs 0.57, aoc-ruc - 0.84 vs 0.83.

In [13]:
# downsample
def downsample(features, target, fraction):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]
    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_downsampled = pd.concat(
        [features_zeros.sample(frac=fraction, random_state=12345)] + [features_ones])
    target_downsampled = pd.concat(
        [target_zeros.sample(frac=fraction, random_state=12345)] + [target_ones])
    
    features_downsampled, target_downsampled = shuffle(
        features_downsampled, target_downsampled, random_state=12345)
    
    return features_downsampled, target_downsampled

features_downsampled, target_downsampled = downsample(features_train, target_train, 0.25)


In [14]:
%%time
model = LogisticRegression(solver='liblinear', random_state=12345)
model.fit(features_downsampled, target_downsampled)
predicted_valid = model.predict(features_valid)
print("F1:", f1_score(target_valid, predicted_valid))
probabilities_valid = model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))

F1: 0.46099290780141844
auc_roc: 0.7162769834594898
CPU times: user 39.3 ms, sys: 59.2 ms, total: 98.6 ms
Wall time: 83.7 ms


In [15]:
%%time
# Дерево

for depth in range(1, 8):
    model_d = DecisionTreeClassifier(max_depth=depth, random_state=12345)  
    model_d.fit(features_downsampled, target_downsampled)
    predictions_valid = model_d.predict(features_valid)
    print("max_depth =", depth, ": ", end='')
#     print(accuracy_score(target_valid, predictions_valid))
    print("F1:", f1_score(target_valid, predictions_valid))
    probabilities_valid = model_d.predict_proba(features_valid)
    probabilities_one_valid = probabilities_valid[:, 1]
    print("auc_roc:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))

max_depth = 1 : F1: 0.4892412231030578
auc_roc: 0.6899243061396131
max_depth = 2 : F1: 0.5187637969094923
auc_roc: 0.7416409681338192
max_depth = 3 : F1: 0.5353418308227114
auc_roc: 0.7954639753294086
max_depth = 4 : F1: 0.4921190893169878
auc_roc: 0.8075992897860012
max_depth = 5 : F1: 0.5758354755784061
auc_roc: 0.8173114662181105
max_depth = 6 : F1: 0.5532359081419624
auc_roc: 0.8333239884122982
max_depth = 7 : F1: 0.5640000000000001
auc_roc: 0.8315792916549856
CPU times: user 119 ms, sys: 61.1 ms, total: 180 ms
Wall time: 188 ms


In [16]:
%%time
# Лес с заданной глубиной

best_model = None
best_result = 0
for est in range(1, 27):
    model_f = RandomForestClassifier(random_state=12345, n_estimators=est)
    model_f.fit(features_downsampled, target_downsampled) 
    predictions_valid = model_f.predict(features_valid)
    result = f1_score(target_valid, predictions_valid) 
    if result > best_result:
        best_model = model_f
        best_result = result

print("F1 наилучшей модели на валидационной выборке:", best_result)
probabilities_valid = best_model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc наилучшей модели:", roc_auc_score(target_valid, probabilities_one_valid, multi_class='ovr'))
print(best_model)

F1 наилучшей модели на валидационной выборке: 0.5784946236559141
auc_roc наилучшей модели: 0.835858330997103
RandomForestClassifier(n_estimators=26, random_state=12345)
CPU times: user 1.6 s, sys: 219 µs, total: 1.6 s
Wall time: 1.61 s


Результаты всех 3-ех моделей в случае upsample идентичны downsample. 

## Тестирование модели

Дерево и Лес дают похожий результат, однако лес в среднем выдает лучший результат

In [17]:
%%time
best_model = None
best_result = 0
for est in range(1, 27):
    model_f = RandomForestClassifier(random_state=12345, n_estimators=est)
    model_f.fit(features_upsampled, target_upsampled) 
    predictions_valid = model_f.predict(features_test)
    result = f1_score(target_test, predictions_valid) 
    if result > best_result:
        best_model = model_f
        best_result = result

print("F1 наилучшей модели на валидационной выборке:", best_result)
probabilities_valid = best_model.predict_proba(features_test)
probabilities_one_valid = probabilities_valid[:, 1]
print("auc_roc наилучшей модели:", roc_auc_score(target_test, probabilities_one_valid, multi_class='ovr'))
print(best_model)

F1 наилучшей модели на валидационной выборке: 0.5924812030075188
auc_roc наилучшей модели: 0.8273297430339159
RandomForestClassifier(n_estimators=25, random_state=12345)
CPU times: user 3.63 s, sys: 12.2 ms, total: 3.64 s
Wall time: 3.65 s


После оценки и обработки данных - работы с пропусками, избавления от ненужных столбцов и добавления дамми переменных для столбцы с текстом я перешел к исследованию. Я разделил наш датасет на 3 части - тренировочная, вариационная и тестируемая выборки. Протестировав 3 модели - лес, дерево и линейная регрессия без учета дисбаланса выборки - линейная регрессия проявила себя хуже всего, оказавшись фактически не рабочей в случае дисбаланса. После исправления дисбаланса выборки, лучше всего себя проявила модель леса. Протестировав ее на лесе, учитывая разные гиперпараметры, лучший полученный результат - f1 = 0.59, auc-roc - 0.83.